<img src="https://i.postimg.cc/J7sf3Lq4/Group.png" alt="FUTURA" width="220" />

# MÓDULO 1 · Sesión 8 — Visualización de datos en Python (v6)
## Matplotlib · Seaborn — **Aprendizaje incremental**: de 1 línea a dashboards profesionales

### Filosofía de esta versión
Cada gráfico se construye en **pasos sucesivos**:

> **Paso 1 → versión mínima** (1 línea de código). 
> **Paso 2 → agregamos un parámetro** (lo explicamos). 
> **Paso 3 → agregamos otro**. 
> **...** 
> **Paso N → versión profesional**.

Esto te permite **enseñar cada parámetro como un concepto nuevo** y que el alumno entienda *por qué* se usa, no solo *qué* hace.

### Estructura
0. Setup + datasets (5 dominios)
1. Anatomía Matplotlib — Figure, Axes, Axis (paso a paso)
2. Construyendo un gráfico de línea profesional (10 pasos)
3. Construyendo un gráfico de barras profesional (8 pasos)
4. De barras simples a barras apiladas (6 pasos)
5. Construyendo un Pareto desde cero (7 pasos)
6. Subplots: de 2 gráficos a un dashboard (6 pasos)
7. Histograma — del básico al avanzado para EDA (8 pasos)
8. Boxplot, Violin y comparación de distribuciones (5 pasos)
9. Heatmaps (correlación + calendario) — paso a paso
10. Validación de modelo: ROC desde cero (6 pasos)
11. Suite completa de validación (clasificación + regresión)
12. Casos sectoriales aplicados
13. Dashboard integrador final
14. Reglas de oro


In [2]:
# ======================================================
# 0) SETUP
# ======================================================
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import matplotlib.dates as mdates
from matplotlib.gridspec import GridSpec

import seaborn as sns

from sklearn.metrics import (
    roc_curve, roc_auc_score,
    precision_recall_curve, average_precision_score,
    confusion_matrix
)
from sklearn.calibration import calibration_curve

rng = np.random.default_rng(7)

pd.set_option("display.max_columns", 200)
sns.set_theme(style="whitegrid", context="notebook")

PALETTE = {
    "navy":  "#034B6F", "teal":  "#00A9E0", "gold":  "#B68A35",
    "green": "#2E933C", "red":   "#C0392B", "grey":  "#6B6B6B",
}
print("Setup listo.")

Setup listo.


## 0.1 Datasets de los 5 dominios


In [2]:
# RETAIL
n = 4000
dates = pd.date_range("2025-01-01", periods=180, freq="D")
products = pd.DataFrame({
    "product": [f"P{str(i).zfill(2)}" for i in range(1, 21)],
    "family":  rng.choice(["Electrónica", "Hogar", "Moda", "Alimentos"], size=20),
    "unit_price": rng.choice([4.9, 9.9, 19.9, 29.9, 49.9, 99.0, 149.0], size=20)
})
weights = rng.dirichlet(np.ones(20) * 0.6)
retail = pd.DataFrame({
    "sale_id": np.arange(1, n + 1),
    "date":    rng.choice(dates, size=n),
    "region":  rng.choice(["Norte", "Centro", "Sur"], size=n, p=[0.35, 0.40, 0.25]),
    "channel": rng.choice(["web", "tienda", "partner"], size=n, p=[0.55, 0.30, 0.15]),
    "product": rng.choice(products["product"], size=n, p=weights),
    "units":   rng.integers(1, 6, size=n),
}).merge(products, on="product", how="left")
season = 1 + 0.12 * np.sin(2 * np.pi * (retail["date"].dt.day / 30))
ch = retail["channel"].map({"web": 1.00, "tienda": 1.06, "partner": 0.95})
discount = rng.choice([0.0, 0.05, 0.10], size=n, p=[0.7, 0.2, 0.1])
retail["revenue"] = (retail["units"] * retail["unit_price"] * season * ch * (1 - discount)).round(2)
retail["cost"]    = (retail["revenue"] * rng.uniform(0.55, 0.78, size=n)).round(2)
retail["margin"]  = (retail["revenue"] - retail["cost"]).round(2)
retail["month"]   = retail["date"].dt.to_period("M").astype(str)
retail["dow"]     = retail["date"].dt.day_name().str[:3]

# BANCA
n_b = 4000
bank = pd.DataFrame({
    "utilization": rng.beta(2, 5, size=n_b),
    "delinq":      rng.poisson(0.4, size=n_b),
    "income":      rng.lognormal(10.2, 0.45, size=n_b),
})
logit = -3.0 + 2.8*bank["utilization"] + 0.7*(bank["delinq"] > 0).astype(int) - 0.000002*bank["income"]
p = 1/(1+np.exp(-logit))
bank["y_true"] = rng.binomial(1, p)
bank["score"]  = np.clip(p + rng.normal(0, 0.06, size=n_b), 0, 1)

# SEGUROS
m = 2200
ins = pd.DataFrame({
    "vehicle_value": rng.lognormal(10.0, 0.35, size=m),
    "claims_12m":    rng.poisson(0.2, size=m),
    "region":        rng.choice(["Norte", "Centro", "Sur"], size=m, p=[0.3, 0.45, 0.25]),
    "driver_age":    rng.integers(18, 75, size=m),
})
true = (300 + 0.0025*ins["vehicle_value"] + 180*ins["claims_12m"] + rng.normal(0, 90, m)).clip(0)
pred = (320 + 0.0022*ins["vehicle_value"] + 140*ins["claims_12m"] + rng.normal(0, 80, m)).clip(0)
ins["y"] = true; ins["yhat"] = pred; ins["resid"] = ins["y"] - ins["yhat"]

# CONTABILIDAD
months = pd.period_range("2025-01", periods=6, freq="M").astype(str)
acct = pd.DataFrame({
    "month":   months,
    "revenue": rng.integers(850_000, 1_050_000, size=6),
    "cogs":    rng.integers(480_000, 620_000,  size=6),
    "opex":    rng.integers(160_000, 240_000,  size=6),
})
acct["profit"] = acct["revenue"] - acct["cogs"] - acct["opex"]

# PRODUCCIÓN
t = pd.date_range("2025-03-01", periods=80, freq="D")
x = rng.normal(10, 0.08, size=len(t)); x[40:48] += 0.22; x[60] = 10.45
prod = pd.DataFrame({"date": t, "diameter": x})

print("Datasets listos.")

Datasets listos.


---
# Bloque 1 — Anatomía de Matplotlib (paso a paso)

## Concepto (pizarra)
Antes de graficar, hay que entender 3 objetos:
- **Figure** = el lienzo
- **Axes** = un gráfico individual dentro del lienzo
- **Axis** = los ejes X e Y dentro de cada `Axes`


### Paso 1 · Crear una Figure y un Axes vacíos


In [ ]:
fig, ax = plt.subplots()
plt.show()
# fig = lienzo. ax = gráfico vacío.

### Paso 2 · Controlar el tamaño con `figsize=(ancho, alto)`
El parámetro está en **pulgadas**. Para reportes ejecutivos usamos típicamente `(8, 4)` o `(10, 5)`.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 3))  # ancho 8, alto 3
plt.show()

### Paso 3 · Etiquetar (lo que distingue un gráfico amateur de uno profesional)
- `set_title(...)` → título del Axes
- `set_xlabel(...)`, `set_ylabel(...)` → nombres de los ejes


In [ ]:
fig, ax = plt.subplots(figsize=(7, 3))
ax.set_title("Esto es el título del Axes")
ax.set_xlabel("Esto es el label del eje X")
ax.set_ylabel("Esto es el label del eje Y")
plt.show()

**Pregunta al grupo:** ¿qué es la Figure y qué es el Axes? ¿qué pasaría si pido `plt.subplots(2, 3)`?


---
# Bloque 2 — Gráfico de línea: 10 pasos hasta lo profesional

**Caso aplicado retail:** evolución del revenue diario.


In [ ]:
daily = retail.groupby("date", as_index=False)["revenue"].sum().sort_values("date")
daily.head()

### Paso 1 · Versión mínima — 1 línea


In [ ]:
fig, ax = plt.subplots()
ax.plot(daily["date"], daily["revenue"])
plt.show()
# Funciona, pero no es presentable.

### Paso 2 · Agregar **título y labels**


In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.5))
ax.plot(daily["date"], daily["revenue"])
ax.set_title("Revenue diario")
ax.set_xlabel("Fecha")
ax.set_ylabel("Revenue")
plt.show()

### Paso 3 · Estilo de la línea: `color`, `linewidth`
- `color` acepta nombres (`"red"`), códigos hex (`"#034B6F"`) o cortos (`"r"`).
- `linewidth` (alias `lw`) controla el grosor en puntos.


In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.5))
ax.plot(daily["date"], daily["revenue"],
        color=PALETTE["navy"], linewidth=1.8)
ax.set_title("Revenue diario")
ax.set_xlabel("Fecha"); ax.set_ylabel("Revenue")
plt.show()

### Paso 4 · Marcadores: `marker`, `markersize`, `markerfacecolor`
- `marker="o"` → puntos circulares en cada dato.
- Otros: `"s"` cuadrado, `"^"` triángulo, `"D"` diamante, `"x"` cruz.


In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.5))
ax.plot(daily["date"], daily["revenue"],
        color=PALETTE["navy"], linewidth=1.6,
        marker="o", markersize=3, markerfacecolor="white")
ax.set_title("Revenue diario — con marcadores")
plt.show()

### Paso 5 · Formato del eje Y como **moneda**
Usamos `mtick.StrMethodFormatter`. La cadena `"${x:,.0f}"` significa:
- `$` → símbolo
- `{x:,.0f}` → el valor `x` con coma cada 3 dígitos y 0 decimales.


In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.5))
ax.plot(daily["date"], daily["revenue"], color=PALETTE["navy"], linewidth=1.6)
ax.set_title("Revenue diario"); ax.set_ylabel("Revenue")
ax.yaxis.set_major_formatter(mtick.StrMethodFormatter("${x:,.0f}"))
plt.show()

### Paso 6 · Formato del eje X como **fecha**
- `MonthLocator()` pone una marca por mes.
- `DateFormatter("%b\n%Y")` muestra mes + año.


In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.5))
ax.plot(daily["date"], daily["revenue"], color=PALETTE["navy"], linewidth=1.6)
ax.set_title("Revenue diario")
ax.yaxis.set_major_formatter(mtick.StrMethodFormatter("${x:,.0f}"))
ax.xaxis.set_major_locator(mdates.MonthLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b\n%Y"))
plt.show()

### Paso 7 · Línea de referencia con `axhline`
Útil para mostrar el **promedio** o un **objetivo de negocio**.


In [ ]:
avg = daily["revenue"].mean()

fig, ax = plt.subplots(figsize=(10, 3.5))
ax.plot(daily["date"], daily["revenue"], color=PALETTE["navy"], linewidth=1.6)
ax.axhline(avg, color=PALETTE["red"], linestyle="--", linewidth=1.2,
           label=f"Promedio: ${avg:,.0f}")
ax.legend()
ax.yaxis.set_major_formatter(mtick.StrMethodFormatter("${x:,.0f}"))
ax.set_title("Revenue diario + promedio")
plt.show()

### Paso 8 · Banda destacada con `axvspan`
Para señalar un periodo (ej. una **campaña** comercial).


In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.5))
ax.plot(daily["date"], daily["revenue"], color=PALETTE["navy"], linewidth=1.6)
ax.axhline(avg, color=PALETTE["red"], linestyle="--", label=f"Promedio: ${avg:,.0f}")
ax.axvspan(pd.Timestamp("2025-04-15"), pd.Timestamp("2025-04-30"),
           color=PALETTE["gold"], alpha=0.15, label="Campaña Q2")
ax.legend()
ax.yaxis.set_major_formatter(mtick.StrMethodFormatter("${x:,.0f}"))
ax.set_title("Revenue diario + campaña destacada")
plt.show()

### Paso 9 · Anotación con flecha (`annotate`)
Tres conceptos:
- `xy=(x, y)` → punto al que apunta la flecha.
- `xytext=(dx, dy)` → desplazamiento del texto.
- `arrowprops` → estilo de la flecha.


In [ ]:
imax = daily["revenue"].idxmax()
x_pico, y_pico = daily.loc[imax, "date"], daily.loc[imax, "revenue"]

fig, ax = plt.subplots(figsize=(10, 3.8))
ax.plot(daily["date"], daily["revenue"], color=PALETTE["navy"], linewidth=1.6)
ax.axhline(avg, color=PALETTE["red"], linestyle="--", label=f"Promedio: ${avg:,.0f}")
ax.annotate(
    f"Pico: ${y_pico:,.0f}",
    xy=(x_pico, y_pico),
    xytext=(20, 25), textcoords="offset points",
    arrowprops=dict(arrowstyle="->", color=PALETTE["red"]),
    bbox=dict(boxstyle="round,pad=0.3", fc="white", ec=PALETTE["red"]),
    fontsize=9, color=PALETTE["red"]
)
ax.legend()
ax.yaxis.set_major_formatter(mtick.StrMethodFormatter("${x:,.0f}"))
ax.set_title("Revenue diario — anotando el pico")
plt.show()

### Paso 10 · Versión profesional final
Juntamos todo + `tight_layout()` + márgenes ajustados.


In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(daily["date"], daily["revenue"], color=PALETTE["navy"], linewidth=1.6,
        marker="o", markersize=3, markerfacecolor="white")

ax.axhline(avg, color=PALETTE["red"], linestyle="--", linewidth=1.2,
           label=f"Promedio diario: ${avg:,.0f}")
ax.axvspan(pd.Timestamp("2025-04-15"), pd.Timestamp("2025-04-30"),
           color=PALETTE["gold"], alpha=0.15, label="Campaña Q2")
ax.annotate(f"Pico: ${y_pico:,.0f}",
            xy=(x_pico, y_pico), xytext=(20, 25), textcoords="offset points",
            arrowprops=dict(arrowstyle="->", color=PALETTE["red"]),
            bbox=dict(boxstyle="round,pad=0.3", fc="white", ec=PALETTE["red"]),
            fontsize=9, color=PALETTE["red"])

ax.set_title("Revenue diario — Retail (versión final)", loc="left", pad=10)
ax.set_xlabel("Fecha"); ax.set_ylabel("Revenue")
ax.yaxis.set_major_formatter(mtick.StrMethodFormatter("${x:,.0f}"))
ax.xaxis.set_major_locator(mdates.MonthLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b\n%Y"))
ax.legend(loc="upper left", frameon=False)
ax.margins(x=0.02)
fig.tight_layout()
plt.show()

**Recapitulando los parámetros aprendidos:**
`figsize`, `color`, `linewidth`, `marker`, `markersize`, `markerfacecolor`, formatter de Y (moneda), formatter de X (fechas), `axhline`, `axvspan`, `annotate` (`xy`, `xytext`, `arrowprops`, `bbox`), `legend(loc, frameon)`, `margins`, `tight_layout`.


---
# Bloque 3 — Gráfico de barras: 8 pasos hasta lo profesional


In [ ]:
fam_rev = retail.groupby("family", as_index=False)["revenue"].sum().sort_values("revenue", ascending=False)
fam_rev

### Paso 1 · Versión mínima


In [ ]:
fig, ax = plt.subplots()
ax.bar(fam_rev["family"], fam_rev["revenue"])
plt.show()

### Paso 2 · Color y ancho con `color` y `width`
`width` está entre 0 y 1. Default es 0.8.


In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.bar(fam_rev["family"], fam_rev["revenue"], color=PALETTE["navy"], width=0.65)
ax.set_title("Revenue por familia")
plt.show()

### Paso 3 · Borde blanco para estilo limpio: `edgecolor` + `linewidth`


In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.bar(fam_rev["family"], fam_rev["revenue"],
       color=PALETTE["navy"], width=0.65,
       edgecolor="white", linewidth=1.5)
ax.set_title("Revenue por familia")
plt.show()

### Paso 4 · Eje Y como moneda


In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.bar(fam_rev["family"], fam_rev["revenue"], color=PALETTE["navy"], width=0.65, edgecolor="white", linewidth=1.5)
ax.yaxis.set_major_formatter(mtick.StrMethodFormatter("${x:,.0f}"))
ax.set_title("Revenue por familia")
plt.show()

### Paso 5 · Etiquetas encima de cada barra con `bar_label`
`bar_label` toma un objeto `BarContainer` (lo que devuelve `ax.bar`). 
`fmt` controla el formato. `padding` separa la etiqueta de la barra.


In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.5))
bars = ax.bar(fam_rev["family"], fam_rev["revenue"], color=PALETTE["navy"], width=0.65, edgecolor="white", linewidth=1.5)
ax.bar_label(bars, fmt="${:,.0f}", padding=3, fontsize=9, color=PALETTE["navy"])
ax.yaxis.set_major_formatter(mtick.StrMethodFormatter("${x:,.0f}"))
ax.set_title("Revenue por familia + etiquetas")
plt.show()

### Paso 6 · Espacio para que las etiquetas no se corten: `set_ylim`


In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.5))
bars = ax.bar(fam_rev["family"], fam_rev["revenue"], color=PALETTE["navy"], width=0.65, edgecolor="white", linewidth=1.5)
ax.bar_label(bars, fmt="${:,.0f}", padding=3, fontsize=9)
ax.yaxis.set_major_formatter(mtick.StrMethodFormatter("${x:,.0f}"))
ax.set_ylim(0, fam_rev["revenue"].max() * 1.18)  # ← deja headroom
ax.set_title("Revenue por familia (con headroom)")
plt.show()

### Paso 7 · Resaltar la barra ganadora con un color distinto
**Truco:** pasar una **lista de colores** a `color`.


In [ ]:
colors = [PALETTE["red"] if i == 0 else PALETTE["navy"] for i in range(len(fam_rev))]

fig, ax = plt.subplots(figsize=(7, 3.5))
bars = ax.bar(fam_rev["family"], fam_rev["revenue"], color=colors, width=0.65, edgecolor="white", linewidth=1.5)
ax.bar_label(bars, fmt="${:,.0f}", padding=3, fontsize=9)
ax.yaxis.set_major_formatter(mtick.StrMethodFormatter("${x:,.0f}"))
ax.set_ylim(0, fam_rev["revenue"].max() * 1.18)
ax.set_title("Revenue por familia — destacando la #1")
plt.show()

### Paso 8 · Versión profesional con todo lo aprendido


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(fam_rev["family"], fam_rev["revenue"],
              color=colors, width=0.65, edgecolor="white", linewidth=1.5)
ax.bar_label(bars, fmt="${:,.0f}", padding=4, fontsize=9, color=PALETTE["navy"])
ax.set_title("Revenue por familia — Retail", loc="left", pad=10)
ax.set_xlabel(""); ax.set_ylabel("Revenue")
ax.yaxis.set_major_formatter(mtick.StrMethodFormatter("${x:,.0f}"))
ax.set_ylim(0, fam_rev["revenue"].max() * 1.18)
ax.spines[["top", "right"]].set_visible(False)
ax.grid(True, axis="y", alpha=0.25)
fig.tight_layout()
plt.show()

---
# Bloque 4 — De barras simples a barras apiladas (6 pasos)

**Caso aplicado:** queremos ver revenue mensual descompuesto por canal.


### Paso 1 · Preparar los datos en formato wide con `pivot_table`
Filas = mes. Columnas = canal. Valores = revenue.


In [ ]:
pivot_mc = retail.pivot_table(index="month", columns="channel", values="revenue",
                              aggfunc="sum", fill_value=0)
pivot_mc

### Paso 2 · Una sola serie (revenue total por mes)


In [ ]:
totals = pivot_mc.sum(axis=1)
fig, ax = plt.subplots(figsize=(9, 3.5))
ax.bar(totals.index, totals.values, color=PALETTE["navy"])
ax.set_title("Revenue mensual total")
plt.show()

### Paso 3 · Tres series **lado a lado** (agrupadas) con `pandas.plot`


In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
pivot_mc.plot(kind="bar", ax=ax,
              color=[PALETTE["navy"], PALETTE["teal"], PALETTE["gold"]])
ax.set_title("Revenue mensual por canal — barras AGRUPADAS")
ax.tick_params(axis="x", rotation=0)
plt.show()

### Paso 4 · Las mismas 3 series **apiladas** con `stacked=True`
**Diferencia clave:** ahora la altura total es el revenue del mes y se ve la composición por canal.


In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
pivot_mc.plot(kind="bar", stacked=True, ax=ax,
              color=[PALETTE["navy"], PALETTE["teal"], PALETTE["gold"]])
ax.set_title("Revenue mensual por canal — APILADAS")
ax.tick_params(axis="x", rotation=0)
plt.show()

### Paso 5 · Lo mismo pero **con `bottom=` manual** (entender qué hace stacked por dentro)


In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
x = np.arange(len(pivot_mc.index))
bottoms = np.zeros(len(pivot_mc.index))
for ch, color in zip(pivot_mc.columns, [PALETTE["navy"], PALETTE["teal"], PALETTE["gold"]]):
    ax.bar(x, pivot_mc[ch], bottom=bottoms, color=color, label=ch, edgecolor="white")
    bottoms += pivot_mc[ch].values
ax.set_xticks(x); ax.set_xticklabels(pivot_mc.index)
ax.set_title("Apilado MANUAL con `bottom=` (lo que hace stacked por dentro)")
ax.legend(title="Canal")
plt.show()

### Paso 6 · **100% apiladas** — composición porcentual
Truco: dividir cada fila por su total.


In [ ]:
pivot_mc_pct = pivot_mc.div(pivot_mc.sum(axis=1), axis=0)

fig, ax = plt.subplots(figsize=(11, 4))
pivot_mc_pct.plot(kind="bar", stacked=True, ax=ax,
                  color=[PALETTE["navy"], PALETTE["teal"], PALETTE["gold"]],
                  edgecolor="white", width=0.85)
ax.set_title("Composición mensual por canal — 100% APILADAS")
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax.set_ylim(0, 1)
ax.tick_params(axis="x", rotation=0)
ax.legend(title="Canal", bbox_to_anchor=(1.02, 1), loc="upper left", frameon=False)
fig.tight_layout()
plt.show()

**Pregunta al grupo:**
- ¿En cuál de los tres formatos se ve mejor la **tendencia del total**?
- ¿En cuál se ve mejor el **cambio de mix**?


---
# Bloque 5 — Pareto desde cero (7 pasos)

**Concepto:** combina barras (cantidad) + línea (% acumulado).
Sirve para responder: ¿qué pocos elementos explican la mayoría?


### Paso 1 · Preparar los datos
Agrupar, sumar y **ordenar de mayor a menor**.


In [ ]:
pareto = (retail.groupby("product", as_index=False)["revenue"].sum()
                .sort_values("revenue", ascending=False).reset_index(drop=True))
pareto.head()

### Paso 2 · Calcular el **% acumulado**


In [ ]:
pareto["cum_pct"] = pareto["revenue"].cumsum() / pareto["revenue"].sum()
pareto.head(8)

### Paso 3 · Solo barras (todavía no es un Pareto)


In [ ]:
fig, ax = plt.subplots(figsize=(11, 3.8))
ax.bar(pareto["product"], pareto["revenue"], color=PALETTE["navy"])
ax.tick_params(axis="x", rotation=45)
ax.set_title("Revenue por producto (ordenado)")
plt.show()

### Paso 4 · Agregar la línea acumulada en el **mismo eje** (mal — no se ve)
Las escalas son muy distintas (~30k vs 0–1).


In [ ]:
fig, ax = plt.subplots(figsize=(11, 3.8))
ax.bar(pareto["product"], pareto["revenue"], color=PALETTE["navy"])
ax.plot(pareto["product"], pareto["cum_pct"], color=PALETTE["red"], marker="o")
ax.tick_params(axis="x", rotation=45)
ax.set_title("❌ Mismo eje — la línea desaparece")
plt.show()

### Paso 5 · Solución: **eje secundario** con `twinx()`
`ax2 = ax.twinx()` crea otro Axes que comparte el eje X.


In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
ax.bar(pareto["product"], pareto["revenue"], color=PALETTE["navy"])
ax.tick_params(axis="x", rotation=45)

ax2 = ax.twinx()
ax2.plot(pareto["product"], pareto["cum_pct"], color=PALETTE["red"], marker="o")
ax2.set_ylim(0, 1.05)
ax.set_title("✅ Con eje secundario — ya se ve la curva acumulada")
plt.show()

### Paso 6 · Línea de referencia 80% + formatos
- `ax2.axhline(0.80)` → la línea horizontal del 80%.
- Format Y izquierdo en moneda, Y derecho en porcentaje.
- Pintar los ticks de cada eje del color correspondiente para no confundir.


In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))

ax.bar(pareto["product"], pareto["revenue"], color=PALETTE["navy"], alpha=0.85)
ax.set_ylabel("Revenue", color=PALETTE["navy"])
ax.tick_params(axis="y", labelcolor=PALETTE["navy"])
ax.yaxis.set_major_formatter(mtick.StrMethodFormatter("${x:,.0f}"))
ax.tick_params(axis="x", rotation=45)

ax2 = ax.twinx()
ax2.plot(pareto["product"], pareto["cum_pct"], color=PALETTE["red"], marker="o", linewidth=2)
ax2.axhline(0.80, color="black", linestyle="--", alpha=0.5)
ax2.text(0.5, 0.81, "80%", color="black", fontsize=9)
ax2.set_ylim(0, 1.05)
ax2.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax2.set_ylabel("Acumulado %", color=PALETTE["red"])
ax2.tick_params(axis="y", labelcolor=PALETTE["red"])
ax2.grid(False)  # evitamos doble grid

ax.set_title("Pareto — Revenue por producto")
fig.tight_layout()
plt.show()

### Paso 7 · Encapsular como **función reutilizable**
Esto es lo que harías en producción: una función que toma el dataframe y devuelve el Pareto.


In [ ]:
def pareto_chart(df_in, cat, value, title="Pareto", top_n=None, ax=None):
    p = (df_in.groupby(cat, as_index=False)[value].sum()
              .sort_values(value, ascending=False).reset_index(drop=True))
    if top_n: p = p.head(top_n)
    p["cum_pct"] = p[value].cumsum() / p[value].sum()

    if ax is None:
        fig, ax = plt.subplots(figsize=(11, 4))
    ax.bar(p[cat], p[value], color=PALETTE["navy"], alpha=0.85)
    ax.set_ylabel(value, color=PALETTE["navy"])
    ax.yaxis.set_major_formatter(mtick.StrMethodFormatter("${x:,.0f}"))
    ax.tick_params(axis="x", rotation=45)
    ax.tick_params(axis="y", labelcolor=PALETTE["navy"])

    ax2 = ax.twinx()
    ax2.plot(p[cat], p["cum_pct"], color=PALETTE["red"], marker="o", linewidth=2)
    ax2.axhline(0.80, color="black", linestyle="--", alpha=0.5)
    ax2.set_ylim(0, 1.05)
    ax2.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
    ax2.set_ylabel("Acumulado %", color=PALETTE["red"])
    ax2.tick_params(axis="y", labelcolor=PALETTE["red"])
    ax2.grid(False)
    ax.set_title(title)
    return (p["cum_pct"] <= 0.80).sum() + 1

n80 = pareto_chart(retail, "product", "revenue", "Pareto — Revenue por producto")
plt.tight_layout(); plt.show()
print(f"Top {n80} productos = ~80% del revenue.")

---
# Bloque 6 — Subplots: de 2 gráficos a un dashboard (6 pasos)


### Paso 1 · `plt.subplots(1, 2)` — el array `axes`
Cuando pedís más de 1 Axes, `plt.subplots` devuelve un **array NumPy**.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
print(type(axes), axes.shape)
axes[0].set_title("Axes 0")
axes[1].set_title("Axes 1")
plt.show()

### Paso 2 · Llenar cada Axes con un gráfico distinto


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
axes[0].plot(daily["date"], daily["revenue"], color=PALETTE["navy"])
axes[0].set_title("Revenue diario")
axes[1].bar(fam_rev["family"], fam_rev["revenue"], color=PALETTE["teal"])
axes[1].set_title("Revenue por familia")
plt.show()

### Paso 3 · Grilla 2×2 — acceder con `axes[i, j]`


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 6))
axes[0, 0].plot(daily["date"], daily["revenue"], color=PALETTE["navy"]); axes[0, 0].set_title("Línea")
axes[0, 1].bar(fam_rev["family"], fam_rev["revenue"], color=PALETTE["teal"]); axes[0, 1].set_title("Barras")
axes[1, 0].hist(retail["revenue"], bins=30, color=PALETTE["gold"]); axes[1, 0].set_title("Histograma")
axes[1, 1].scatter(retail["revenue"], retail["margin"], alpha=0.3, color=PALETTE["green"]); axes[1, 1].set_title("Scatter")
fig.tight_layout()
plt.show()

### Paso 4 · Compartir ejes con `sharex` / `sharey`
Cuando los gráficos comparten escala, ayuda a comparar.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.5), sharey=True)
for ax, region in zip(axes, ["Norte", "Centro", "Sur"]):
    sub = retail[retail["region"] == region].groupby("month")["revenue"].sum()
    ax.bar(sub.index, sub.values, color=PALETTE["navy"])
    ax.set_title(region); ax.tick_params(axis="x", rotation=30)
axes[0].set_ylabel("Revenue")
fig.suptitle("Comparación regional — eje Y compartido", fontweight="bold")
fig.tight_layout()
plt.show()

### Paso 5 · Tamaños distintos con `gridspec_kw`
`width_ratios` y `height_ratios` controlan el tamaño relativo de columnas y filas.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 5),
                         gridspec_kw={"width_ratios": [3, 1], "height_ratios": [2, 1]})
for i, ax in enumerate(axes.ravel(), start=1):
    ax.set_title(f"Axes #{i}")
fig.suptitle("width_ratios=[3,1] · height_ratios=[2,1]", fontweight="bold")
fig.tight_layout()
plt.show()

### Paso 6 · `subplot_mosaic` — la forma moderna y declarativa
Cada letra define un panel; letras repetidas → ese panel ocupa varias celdas.


In [ ]:
fig, axd = plt.subplot_mosaic(
    """
    AAB
    CCB
    """,
    figsize=(11, 5),
    gridspec_kw={"hspace": 0.4, "wspace": 0.3}
)
axd["A"].plot(daily["date"], daily["revenue"], color=PALETTE["navy"]); axd["A"].set_title("A — Línea")
axd["B"].bar(fam_rev["family"], fam_rev["revenue"], color=PALETTE["gold"]); axd["B"].set_title("B — Barras")
axd["C"].hist(retail["revenue"], bins=25, color=PALETTE["teal"]); axd["C"].set_title("C — Histograma")
fig.suptitle("subplot_mosaic — sintaxis declarativa", fontweight="bold")
plt.show()

---
# Bloque 7 — Histograma para EDA (8 pasos)

**Caso aplicado banca:** distribución del score de un modelo de default.


### Paso 1 · Versión mínima


In [ ]:
fig, ax = plt.subplots()
ax.hist(bank["score"])
plt.show()

### Paso 2 · Controlar el número de bins (`bins=`)
Pocos bins → pierdo detalle. Demasiados → ruido.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3))
for ax, b in zip(axes, [10, 30, 100]):
    ax.hist(bank["score"], bins=b, color=PALETTE["navy"], edgecolor="white")
    ax.set_title(f"bins={b}")
plt.show()

### Paso 3 · Pasar a Seaborn — más expresivo


In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.5))
sns.histplot(bank["score"], bins=30, ax=ax, color=PALETTE["navy"])
ax.set_title("sns.histplot básico")
plt.show()

### Paso 4 · Agregar la **densidad estimada** con `kde=True`


In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.5))
sns.histplot(bank["score"], bins=30, kde=True, ax=ax, color=PALETTE["navy"])
ax.set_title("Histograma + KDE")
plt.show()

### Paso 5 · Separar por categoría con `hue=`
**Caso DS:** ¿el score separa visualmente a los que defaultearon de los que no?


In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.8))
sns.histplot(data=bank, x="score", hue="y_true", bins=30, ax=ax,
             palette=[PALETTE["navy"], PALETTE["red"]])
ax.set_title("Score por clase (overlap)")
plt.show()

### Paso 6 · Comparar **forma** de las distribuciones — `stat="density"` + `common_norm=False`
Sin esto, la clase mayoritaria aplasta visualmente a la minoritaria.


In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.8))
sns.histplot(data=bank, x="score", hue="y_true",
             stat="density", common_norm=False,
             element="step", bins=30, alpha=0.4, ax=ax,
             palette=[PALETTE["navy"], PALETTE["red"]])
ax.set_title("Densidad por grupo (común para drift y validación de score)")
plt.show()

### Paso 7 · Agregar línea vertical del **umbral de decisión**


In [ ]:
thr = 0.35
fig, ax = plt.subplots(figsize=(9, 3.8))
sns.histplot(data=bank, x="score", hue="y_true",
             stat="density", common_norm=False, element="step",
             bins=30, alpha=0.4, ax=ax,
             palette=[PALETTE["navy"], PALETTE["red"]])
ax.axvline(thr, color="black", linestyle="--", label=f"umbral = {thr}")
ax.legend()
ax.set_title("Densidad por grupo + umbral de decisión")
plt.show()

### Paso 8 · Versión final con anotación del trade-off


In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.2))
sns.histplot(data=bank, x="score", hue="y_true",
             stat="density", common_norm=False, element="step",
             bins=30, alpha=0.4, ax=ax,
             palette=[PALETTE["navy"], PALETTE["red"]])
ax.axvline(thr, color="black", linestyle="--", label=f"umbral = {thr}")
ax.annotate("FN\n(default no detectado)", xy=(0.20, 0.5), color=PALETTE["red"], fontsize=9, ha="center")
ax.annotate("FP\n(buen cliente rechazado)", xy=(0.55, 0.8), color=PALETTE["navy"], fontsize=9, ha="center")
ax.legend()
ax.set_title("Score del modelo de default — visualizando el trade-off")
fig.tight_layout()
plt.show()

---
# Bloque 8 — Boxplot, Violin y comparación de distribuciones (5 pasos)


### Paso 1 · Boxplot básico


In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.5))
sns.boxplot(data=retail, x="family", y="revenue", ax=ax)
ax.set_title("Boxplot básico — revenue por familia")
plt.show()

### Paso 2 · Paleta consistente con `palette=`


In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.5))
sns.boxplot(data=retail, x="family", y="revenue", ax=ax, palette="Blues")
ax.set_title("Box + paleta Blues")
plt.show()

### Paso 3 · Separar por una segunda categoría con `hue=`


In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
sns.boxplot(data=retail, x="family", y="revenue", hue="region",
            palette="Blues", ax=ax)
ax.set_title("Box por familia, separado por región (hue)")
plt.show()

### Paso 4 · Saltar a violin para ver la **forma** completa
`inner='quartile'` dibuja líneas en los cuartiles.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
sns.violinplot(data=retail, x="channel", y="revenue",
               inner="quartile", palette="Blues", ax=ax)
ax.set_title("Violin — distribución completa por canal")
plt.show()

### Paso 5 · Box + Violin lado a lado para la misma comparación


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.boxplot(data=retail, x="channel", y="revenue", palette="Blues", ax=axes[0])
axes[0].set_title("Box (resumen estadístico)")
sns.violinplot(data=retail, x="channel", y="revenue", inner="quartile", palette="Blues", ax=axes[1])
axes[1].set_title("Violin (forma completa)")
fig.suptitle("Box vs Violin — el mismo dato visto de dos formas", fontweight="bold")
fig.tight_layout()
plt.show()

---
# Bloque 9 — Heatmaps paso a paso (correlación + calendario)


## 9.1 Heatmap de correlaciones

### Paso 1 · Matriz cruda


In [ ]:
num = retail[["units", "unit_price", "revenue", "cost", "margin"]].dropna()
corr = num.corr()
corr.round(2)

### Paso 2 · Heatmap básico


In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
sns.heatmap(corr, ax=ax)
ax.set_title("Heatmap básico (sin números)")
plt.show()

### Paso 3 · Anotar valores con `annot=True` y formato `fmt=".2f"`


In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
sns.heatmap(corr, annot=True, fmt=".2f", ax=ax)
ax.set_title("Con valores anotados")
plt.show()

### Paso 4 · Cambiar paleta y centrar en 0 — `cmap='coolwarm'`, `center=0`
Para correlaciones, una paleta divergente con centro en 0 deja claro qué es positivo y qué negativo.


In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, vmin=-1, vmax=1, ax=ax)
ax.set_title("Paleta divergente centrada en 0")
plt.show()

### Paso 5 · Ocultar la mitad redundante con `mask=`
Una matriz de correlación es simétrica → mostramos solo la triangular inferior.


In [ ]:
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="coolwarm",
            center=0, vmin=-1, vmax=1, linewidths=0.5, linecolor="white",
            cbar_kws={"label": "Pearson", "shrink": 0.8}, ax=ax)
ax.set_title("Heatmap correlación — versión final")
fig.tight_layout()
plt.show()

## 9.2 Heatmap calendario (estacionalidad)

### Paso 1 · Construir la tabla pivotada


In [ ]:
tmp = retail.copy()
tmp["week_of_month"] = ((tmp["date"].dt.day - 1) // 7) + 1
order_dow = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
ht = tmp.pivot_table(index="dow", columns="week_of_month",
                     values="revenue", aggfunc="sum").reindex(order_dow)
ht

### Paso 2 · Heatmap básico → versión profesional


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
sns.heatmap(ht, annot=True, fmt=".0f", cmap="Blues",
            linewidths=0.4, linecolor="white",
            cbar_kws={"label": "Revenue", "shrink": 0.8}, ax=ax)
ax.set_title("Estacionalidad — Revenue por día y semana del mes")
ax.set_xlabel("Semana del mes"); ax.set_ylabel("Día")
fig.tight_layout()
plt.show()

---
# Bloque 10 — Curva ROC desde cero (6 pasos)

## Concepto
La curva ROC muestra cómo cambia **TPR (sensibilidad)** vs **FPR (falsos positivos)** al variar el umbral de decisión. 
El **AUC** (área bajo la curva) resume su calidad: 0.5 = aleatorio, 1.0 = perfecto.


### Paso 1 · Calcular FPR, TPR y AUC con sklearn


In [ ]:
fpr, tpr, _ = roc_curve(bank["y_true"], bank["score"])
auc = roc_auc_score(bank["y_true"], bank["score"])
print("AUC =", round(auc, 3))

### Paso 2 · Plot básico


In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
ax.plot(fpr, tpr)
ax.set_title("ROC básica")
plt.show()

### Paso 3 · Etiquetas correctas + diagonal de referencia (modelo aleatorio)


In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
ax.plot(fpr, tpr, color=PALETTE["navy"], linewidth=2)
ax.plot([0, 1], [0, 1], "--", color="grey", label="Aleatorio")
ax.set_xlabel("FPR (falsos positivos)")
ax.set_ylabel("TPR (sensibilidad)")
ax.set_title("ROC + diagonal")
ax.legend()
plt.show()

### Paso 4 · Mostrar el AUC en la leyenda


In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
ax.plot(fpr, tpr, color=PALETTE["navy"], linewidth=2, label=f"AUC = {auc:.3f}")
ax.plot([0, 1], [0, 1], "--", color="grey", label="Aleatorio")
ax.set_xlabel("FPR"); ax.set_ylabel("TPR")
ax.set_title("ROC con AUC")
ax.legend(loc="lower right")
plt.show()

### Paso 5 · Sombrear el área bajo la curva con `fill_between`


In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
ax.plot(fpr, tpr, color=PALETTE["navy"], linewidth=2, label=f"AUC = {auc:.3f}")
ax.fill_between(fpr, tpr, alpha=0.15, color=PALETTE["navy"])
ax.plot([0, 1], [0, 1], "--", color="grey", label="Aleatorio")
ax.set_xlabel("FPR"); ax.set_ylabel("TPR")
ax.set_title("ROC con AUC sombreado")
ax.legend(loc="lower right")
plt.show()

### Paso 6 · Marcar el punto operativo (umbral elegido)


In [ ]:
# Para un umbral elegido, calcular el (FPR, TPR) correspondiente
yhat = (bank["score"] >= thr).astype(int)
tn, fp, fn, tp = confusion_matrix(bank["y_true"], yhat).ravel()
op_fpr, op_tpr = fp/(fp+tn), tp/(tp+fn)

fig, ax = plt.subplots(figsize=(5, 5))
ax.plot(fpr, tpr, color=PALETTE["navy"], linewidth=2, label=f"AUC = {auc:.3f}")
ax.fill_between(fpr, tpr, alpha=0.15, color=PALETTE["navy"])
ax.plot([0, 1], [0, 1], "--", color="grey", label="Aleatorio")
ax.scatter([op_fpr], [op_tpr], color=PALETTE["red"], s=80, zorder=5,
           label=f"Umbral={thr} → TPR={op_tpr:.2f}, FPR={op_fpr:.2f}")
ax.set_xlabel("FPR"); ax.set_ylabel("TPR")
ax.set_title("ROC final con punto operativo")
ax.legend(loc="lower right", fontsize=8)
plt.show()

---
# Bloque 11 — Suite completa de validación

Ya entendemos cada gráfico individualmente — ahora los combinamos en un panel único.


In [ ]:
prec, rec, _ = precision_recall_curve(bank["y_true"], bank["score"])
ap = average_precision_score(bank["y_true"], bank["score"])
cm = confusion_matrix(bank["y_true"], (bank["score"] >= thr).astype(int))
prob_true, prob_pred = calibration_curve(bank["y_true"], bank["score"], n_bins=10, strategy="quantile")

fig, axd = plt.subplot_mosaic(
    """
    AB
    CD
    EE
    """, figsize=(13, 11),
    gridspec_kw={"hspace": 0.45, "wspace": 0.3}
)

# A: KDE score por clase
sns.kdeplot(data=bank, x="score", hue="y_true", fill=True, common_norm=False,
            alpha=0.4, palette=[PALETTE["navy"], PALETTE["red"]], ax=axd["A"])
axd["A"].axvline(thr, color="black", linestyle="--")
axd["A"].set_title("A. Score por clase")

# B: ROC
axd["B"].plot(fpr, tpr, color=PALETTE["navy"], linewidth=2, label=f"AUC = {auc:.3f}")
axd["B"].plot([0, 1], [0, 1], "--", color="grey")
axd["B"].fill_between(fpr, tpr, alpha=0.15, color=PALETTE["navy"])
axd["B"].set_title("B. ROC"); axd["B"].legend(loc="lower right")

# C: PR
axd["C"].plot(rec, prec, color=PALETTE["green"], linewidth=2, label=f"AP = {ap:.3f}")
axd["C"].axhline(bank["y_true"].mean(), color="grey", linestyle="--", label="Baseline")
axd["C"].set_title("C. Precision-Recall"); axd["C"].legend()

# D: Confusion matrix
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False,
            xticklabels=["No def", "Def"], yticklabels=["No def", "Def"], ax=axd["D"])
axd["D"].set_title(f"D. Matriz confusión (umbral={thr})")
axd["D"].set_xlabel("Predicho"); axd["D"].set_ylabel("Real")

# E: Calibration
axd["E"].plot(prob_pred, prob_true, marker="o", color=PALETTE["navy"], linewidth=2, label="Modelo")
axd["E"].plot([0, 1], [0, 1], "--", color="grey", label="Calibrado perfecto")
axd["E"].set_title("E. Calibration curve")
axd["E"].set_xlabel("Probabilidad predicha"); axd["E"].set_ylabel("Frecuencia real")
axd["E"].legend()

fig.suptitle("Banca — Suite completa de validación", fontsize=15, fontweight="bold")
plt.show()

## 11.1 Lift y Gains chart (negocio: ¿cuánto vale el modelo?)


In [ ]:
tmp = bank.sort_values("score", ascending=False).reset_index(drop=True)
tmp["decile"] = pd.qcut(-tmp["score"], q=10, labels=False) + 1
lift = tmp.groupby("decile").agg(n=("y_true", "size"), pos=("y_true", "sum")).reset_index()
rate = tmp["y_true"].mean()
lift["lift"]      = (lift["pos"]/lift["n"]) / rate
lift["cum_share"] = lift["pos"].cumsum() / tmp["y_true"].sum()
lift["cum_pop"]   = lift["n"].cumsum() / lift["n"].sum()

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].bar(lift["decile"], lift["lift"], color=PALETTE["navy"], edgecolor="white")
axes[0].axhline(1, color="black", linestyle="--", label="Aleatorio")
axes[0].set_title("Lift por decil"); axes[0].legend()

axes[1].plot([0]+list(lift["cum_pop"]), [0]+list(lift["cum_share"]),
             marker="o", color=PALETTE["navy"], linewidth=2.2, label="Modelo")
axes[1].plot([0, 1], [0, 1], "--", color="grey", label="Aleatorio")
axes[1].set_title("Gains chart")
axes[1].xaxis.set_major_formatter(mtick.PercentFormatter(1.0))
axes[1].yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
axes[1].legend()

fig.suptitle("Lift & Gains — utilidad del modelo en negocio", fontweight="bold")
fig.tight_layout()
plt.show()

## 11.2 Diagnóstico de regresión (seguros)


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

sns.scatterplot(data=ins.sample(700, random_state=0), x="yhat", y="y",
                hue="region", alpha=0.6, ax=axes[0])
lims = [ins["y"].min(), ins["y"].max()]
axes[0].plot(lims, lims, "k--", alpha=0.6); axes[0].set_title("y vs ŷ")

sns.histplot(ins["resid"], bins=40, kde=True, color=PALETTE["navy"], ax=axes[1])
axes[1].axvline(0, color="black"); axes[1].set_title("Distribución de residuales")

sns.scatterplot(data=ins.sample(700, random_state=1), x="yhat", y="resid",
                alpha=0.5, color=PALETTE["teal"], ax=axes[2])
axes[2].axhline(0, color="black")
axes[2].set_title("Residuos vs ŷ")

fig.suptitle("Seguros — Diagnóstico visual de regresión", fontweight="bold")
fig.tight_layout()
plt.show()

---
# Bloque 12 — Casos sectoriales aplicados


## 12.1 Producción — Control chart (SPC)
Combina varios elementos vistos: línea + `axhline` (centro, UCL, LCL) + `axhspan` (zona ±1σ) + scatter destacado.


In [ ]:
c, s = prod["diameter"].mean(), prod["diameter"].std()
ucl, lcl = c + 3*s, c - 3*s
ooc = prod[(prod["diameter"] > ucl) | (prod["diameter"] < lcl)]

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(prod["date"], prod["diameter"], marker="o", linewidth=1.3, markersize=4, color=PALETTE["navy"])
ax.axhline(c,   color=PALETTE["navy"], linewidth=2,  label="Centro")
ax.axhline(ucl, color=PALETTE["red"],  linestyle="--", label="UCL +3σ")
ax.axhline(lcl, color=PALETTE["red"],  linestyle="--", label="LCL −3σ")
ax.axhspan(c - s, c + s, color=PALETTE["green"], alpha=0.08, label="±1σ")
ax.scatter(ooc["date"], ooc["diameter"], color=PALETTE["red"], s=90, zorder=5, label="Fuera de control")
ax.set_title("Producción — Control chart (SPC)"); ax.set_ylabel("Diámetro (mm)")
ax.legend(ncol=5, loc="upper left", fontsize=8)
fig.tight_layout(); plt.show()

## 12.2 Contabilidad — P&L con Profit MoM y semáforo


In [ ]:
acct["mom"] = acct["profit"].pct_change()
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(acct["month"], acct["profit"], marker="o", linewidth=2, color=PALETTE["navy"])
axes[0].fill_between(acct["month"], acct["profit"], alpha=0.15, color=PALETTE["navy"])
axes[0].set_title("Profit mensual")
axes[0].yaxis.set_major_formatter(mtick.StrMethodFormatter("${x:,.0f}"))

colors_mom = [PALETTE["green"] if v >= 0 else PALETTE["red"] for v in acct["mom"].fillna(0)]
bars = axes[1].bar(acct["month"], acct["mom"], color=colors_mom, edgecolor="white")
axes[1].axhline(0, color="black")
axes[1].bar_label(bars, fmt="{:.1%}", padding=3, fontsize=9)
axes[1].yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
axes[1].set_title("Variación MoM (semáforo)")

fig.tight_layout(); plt.show()

## 12.3 Retail — Volumen por producto apilado por familia


In [ ]:
vol = retail.pivot_table(index="product", columns="family",
                         values="units", aggfunc="sum", fill_value=0)
vol = vol.loc[vol.sum(axis=1).sort_values(ascending=False).index]

fig, ax = plt.subplots(figsize=(12, 4))
vol.plot(kind="bar", stacked=True, ax=ax,
         color=[PALETTE["navy"], PALETTE["teal"], PALETTE["gold"], PALETTE["green"]],
         edgecolor="white", linewidth=0.8)
ax.set_title("Volumen de unidades por producto — apilado por familia")
ax.tick_params(axis="x", rotation=45)
ax.legend(title="Familia", bbox_to_anchor=(1.02, 1), loc="upper left", frameon=False)
fig.tight_layout(); plt.show()

---
# Bloque 13 — Dashboard integrador final

Todo lo que aprendimos paso a paso, ahora combinado.


In [ ]:
kpi_month = (retail.groupby("month", as_index=False)
                   .agg(revenue=("revenue", "sum"), margin=("margin", "sum")))
kpi_month["margin_pct"] = kpi_month["margin"] / kpi_month["revenue"]

fig, axd = plt.subplot_mosaic(
    """
    AAB
    CCB
    DDE
    """,
    figsize=(15, 9),
    gridspec_kw={"hspace": 0.55, "wspace": 0.35}
)

# A — Revenue mensual
axd["A"].plot(kpi_month["month"], kpi_month["revenue"], marker="o", linewidth=2, color=PALETTE["navy"])
axd["A"].fill_between(kpi_month["month"], kpi_month["revenue"], alpha=0.15, color=PALETTE["navy"])
axd["A"].set_title("A. Revenue mensual")
axd["A"].yaxis.set_major_formatter(mtick.StrMethodFormatter("${x:,.0f}"))

# B — Donut familia
fam = retail.groupby("family")["revenue"].sum().sort_values(ascending=False)
axd["B"].pie(fam, labels=fam.index, autopct="%1.0f%%",
             colors=[PALETTE["navy"], PALETTE["teal"], PALETTE["gold"], PALETTE["green"]],
             wedgeprops=dict(width=0.4, edgecolor="white"))
axd["B"].set_title("B. Mix familia")

# C — Apilado mensual canal
pivot_mc.plot(kind="bar", stacked=True, ax=axd["C"],
              color=[PALETTE["navy"], PALETTE["teal"], PALETTE["gold"]],
              edgecolor="white", width=0.85)
axd["C"].set_title("C. Revenue mensual apilado por canal")
axd["C"].yaxis.set_major_formatter(mtick.StrMethodFormatter("${x:,.0f}"))
axd["C"].tick_params(axis="x", rotation=0)
axd["C"].legend(title="Canal", fontsize=8)

# D — Pareto top 12
pareto_chart(retail, "product", "revenue", "D. Pareto top 12 productos", top_n=12, ax=axd["D"])

# E — ROC modelo
axd["E"].plot(fpr, tpr, color=PALETTE["navy"], linewidth=2, label=f"AUC={auc:.2f}")
axd["E"].plot([0, 1], [0, 1], "--", color="grey")
axd["E"].fill_between(fpr, tpr, alpha=0.15, color=PALETTE["navy"])
axd["E"].set_title("E. ROC modelo riesgo"); axd["E"].legend()

fig.suptitle("Dashboard integrador — Negocio + Modelo", fontsize=15, fontweight="bold")
plt.show()

---
# Bloque 14 — Reglas de oro

## Aprendizaje incremental (lo que hicimos hoy)
1. **Empezar mínimo, agregar de a un parámetro.** Si entendés cada parámetro, el gráfico complejo es solo la suma de varios pasos simples.
2. **Cada parámetro responde a un problema concreto.** `figsize` → tamaño físico; `bottom` → apilar; `twinx` → dos escalas; `mask` → ocultar redundancia.

## Para Data Science
1. EDA primero (distribuciones, correlaciones, missingness).
2. Comparar **densidades por grupo** con `common_norm=False` (drift y validación).
3. Suite de validación: **KDE + ROC + PR + CM + Calibration + Lift/Gains**.
4. Regresión: **y vs ŷ + residuales + residuales vs ŷ**.

## Para Negocio
1. Apilado / 100% / Pareto para **mix y prioridad**.
2. Heatmap calendario para **estacionalidad**.
3. Funnel para **conversión**, Waterfall para **descomposición**, Cohort para **retención**.
4. Anotar siempre el insight clave (con `annotate`).

## Para presentar como mentor
1. Siempre OO (`fig, ax = plt.subplots()`).
2. Etiquetar todo: título, ejes, unidades, leyenda.
3. Color codifica una variable, no decora.
4. `tight_layout()` o `constrained_layout=True`.
5. Exportar con `savefig(dpi=200, bbox_inches='tight')`.

## Práctica sugerida
Con un dataset propio, replicá esta progresión:
1. Línea KPI (10 pasos)
2. Barras profesionales (8 pasos)
3. Apiladas + 100% (6 pasos)
4. Pareto (7 pasos)
5. Validación de tu modelo (ROC desde cero)
6. Dashboard final con `subplot_mosaic`
